In [2]:
import torch
import torch.nn as nn

In [3]:
patch_tokens = torch.randn(1, 256, 768)

cls_token = torch.randn(1, 768)

print(patch_tokens.shape)

torch.Size([1, 256, 768])


In [4]:
class SpatialConsistency(nn.Module):

    def __init__(self):

        super().__init__()

        self.attn = nn.MultiheadAttention(embed_dim=768, num_heads=8, batch_first=True)

    def forward(self, x):

        out, _ = self.attn(x, x, x)

        return out

In [5]:
class FeatureConsistency(nn.Module):

    def __init__(self):

        super().__init__()

        self.block = nn.Sequential(nn.Linear(768, 768), nn.GELU(), nn.LayerNorm(768))

    def forward(self, x):

        return self.block(x)

In [6]:
class SemanticConsistency(nn.Module):

    def __init__(self):

        super().__init__()

        self.attn = nn.MultiheadAttention(embed_dim=768, num_heads=8, batch_first=True)

    def forward(self, patches, cls_token):

        cls_token = cls_token.unsqueeze(1)

        out, _ = self.attn(patches, cls_token, cls_token)

        return out

In [7]:
class FusionBlock(nn.Module):

    def __init__(self):

        super().__init__()

        self.proj = nn.Linear(2304, 768)

    def forward(self, s, f, m):

        x = torch.cat([s, f, m], dim=-1)

        return self.proj(x)

In [8]:
class ThreeCModule(nn.Module):

    def __init__(self):

        super().__init__()

        self.spatial = SpatialConsistency()

        self.feature = FeatureConsistency()

        self.semantic = SemanticConsistency()

        self.fusion = FusionBlock()

    def forward(self, patches, cls_token):

        s = self.spatial(patches)

        f = self.feature(patches)

        m = self.semantic(patches, cls_token)

        out = self.fusion(s, f, m)

        return out

In [9]:
threec = ThreeCModule()

enhanced_tokens = threec(patch_tokens, cls_token)

print(enhanced_tokens.shape)

torch.Size([1, 256, 768])


In [14]:
threec = ThreeCModule()

enhanced_tokens = threec(patch_tokens, cls_token)

print(enhanced_tokens.shape)

torch.Size([1, 256, 768])
